# `c10_f` — Finance Under Three Accounting Standards

**Component curation notebook.** Fetches the raw IPEDS distribution files, verifies the
reference period against official documentation, locks the schema, reshapes to the
declared grain, validates, and writes one curated table with a metadata sidecar.

| Property | Value |
|---|---|
| Native tables | `F2223_F1A`, `F2223_F2`, `F2223_F3`, `DRVF2023` |
| Reference period | Fiscal year 2023 |
| Curated grain | `UNITID` |
| Output | `data/curated/c10_f.parquet` |

Three mutually incompatible forms cover the sector: F1A for GASB public institutions, F2 for FASB private not-for-profit institutions and a small number of FASB-reporting publics, and F3 for private for-profit institutions. Route each UNITID by CONTROL from the c01 spine, harmonise onto common concepts, and keep a reporting_standard column on every row so no downstream comparison can silently cross the standards.

> **Pitfall.** The same economic concept has different variable names and sometimes different definitions on each form. Comparing a GASB expense total directly against a FASB one is the single most common serious error in IPEDS finance work. Prefer the derived file DRVF2023 for cross-sector comparison, because NCES has already done the harmonisation there, and use the raw forms when a specific line item is needed that the derived file does not carry.

## 1. Environment

One import surface, so a parsing quirk is fixed once rather than twelve times.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import ipeds_utils as iu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

SLUG = "c10_f"
TABLES = ['F2223_F1A', 'F2223_F2', 'F2223_F3', 'DRVF2023']
GRAIN = ['UNITID']
REFERENCE_PERIOD = 'Fiscal year 2023'

print("ipeds_utils", iu.__version__, "| pandas", pd.__version__)

ipeds_utils 1.1.0 | pandas 3.0.5


## 2. Retrieve

Downloads are cached, so re-running this notebook is offline and cheap. Every retrieval returns a provenance record carrying a SHA-256 digest, which is what makes a result reproducible rather than merely repeatable.

In [2]:
RAW_DIR = "../data/raw"   # relative to notebooks/, so all twelve share one cache

provenance = [iu.fetch(t, raw_dir=RAW_DIR) for t in TABLES]
pd.DataFrame(provenance)[["table", "data_bytes", "data_sha256", "retrieved_utc"]]

,table,data_bytes,data_sha256,retrieved_utc
0,F2223_F1A,711679,beabdb0384f000a2fa487e7083512f880013c897d8e10b...,2026-09-24T17:19:33+00:00
1,F2223_F2,560124,87d7d5768903d9dda56126822f6baca95f483b232ff20e...,2026-09-24T17:19:33+00:00
2,F2223_F3,228105,b27954fb76b97e40a97b392c81a2cecd935f06058222ea...,2026-09-24T17:19:33+00:00
3,DRVF2023,364081,629c1c9efb2c286d6e157551428223060b80ee09375919...,2026-09-24T17:19:33+00:00


## 3. Verify the reference period

**Do not skip this cell.** The filename year is not the reference period, and the offsets are not uniform across components. This assertion fails loudly rather than letting a misaligned period corrupt every downstream year comparison, where it would be invisible in the data itself.

In [3]:
intro = iu.assert_reference_period(
    provenance[0]["dict_path"],
    expect=r'(2022-23|fiscal year 2023|FY2023)',
    table=TABLES[0],
)
print(intro[:600])

File documentation public institutions - GASB 34/35: Fiscal year 2023
(Provisional release)
Filename F2223_F1A
Overview This data file contains institutional finance data for public institutions that use accounting standards established by the Governmental Accounting Standards Board (GASB).  Finance data includes institutional revenues by source, expenditures by functional categories, expenditures by natural classification categories, scholarships and fellowships by source, endowments, assets and liabilities. This information provides context for understanding the cost and resources of providi


## 4. Inspect the dictionary

Variable labels come from the published dictionary, never from memory. This is also where value sets are read, so categorical decoding is driven by the official codebook and a taxonomy revision surfaces as unmatched codes instead of a plausible-looking wrong label.

In [4]:
variables = iu.read_dict(provenance[0]["dict_path"])
valuesets = iu.read_valuesets(provenance[0]["dict_path"])

print(f"{len(variables)} variables documented, {len(valuesets)} value-set rows")
variables[["varname", "vartitle"]].head(20)

146 variables documented, 8 value-set rows


,varname,vartitle
0,UNITID,Unique identification number of the institution
1,F1A01,Total current assets
2,F1A31,"Depreciable capital assets, net of depreciation"
3,F1A04,Other noncurrent assets
4,F1A05,Total noncurrent assets
5,F1A06,Total assets
6,F1A19,Deferred outflows of resources
7,F1A07,"Long-term debt, current portion"
8,F1A08,Other current liabilities
9,F1A09,Total current liabilities


## 5. Load and lock the schema

The first run records the column signature; later runs fail if it drifts.

In [5]:
KEEP = ['UNITID']

raw = iu.read_csv(provenance[0]["data_path"])
print("raw shape", raw.shape)

lock = iu.lock_schema(raw, TABLES[0], schema_dir="../schemas", strict=False)
print("schema:", lock["status"], "| added", lock["added"][:5], "| removed", lock["removed"][:5])

available = [c for c in KEEP if c in raw.columns]
missing = [c for c in KEEP if c not in raw.columns]
if missing:
    print("NOT PRESENT in this cycle (verify against the varlist above):", missing)

frame = raw[available].copy()
frame.head()

raw shape

 (1916, 288)
schema: unchanged | added [] | removed []


,UNITID
0,100654
1,100663
2,100706
3,100724
4,100733


## 6. Mask reserved missing codes

IPEDS encodes missingness as negative integers. A mean computed without masking them is badly wrong and looks entirely plausible, which is what makes this the most costly single omission in IPEDS analysis.

In [6]:
RESERVED = [-1, -2, -3, -9]

numeric_cols = [
    c for c in frame.columns
    if c not in ("UNITID", *GRAIN) and pd.api.types.is_numeric_dtype(frame[c])
]

before = frame[numeric_cols].isna().sum().sum()
for col in numeric_cols:
    frame.loc[frame[col].isin(RESERVED), col] = np.nan
after = frame[numeric_cols].isna().sum().sum()

# Masking turns an integer column into float (1 becomes 1.0). Measures can stay float,
# since NaN is what the models expect, but category codes go back to nullable integers
# so they print, join, and decode as codes rather than as 1.0.
for col in []:
    if col in frame.columns and pd.api.types.is_float_dtype(frame[col]):
        if (frame[col].dropna() % 1 == 0).all():
            frame[col] = frame[col].astype("Int64")

print(f"masked {after - before:,} reserved-code cells across {len(numeric_cols)} numeric columns")

masked 0.0 reserved-code cells across 0 numeric columns


## 7. Carry the imputation flags

An imputed value and a reported value are not the same evidence. A column where most institutions carry a generated flag should not be modelled as though it were observed, and this is where that judgement becomes possible.

In [7]:
values, flags = iu.split_imputation_flags(raw, numeric_cols)

if flags.shape[1] > 1:
    summary = iu.imputation_summary(flags)
    display(summary.head(15))
    reported = summary[summary.flag == "R"].set_index("column")["share"]
    weak = reported[reported < 0.90]
    if len(weak):
        print("Columns under 90% reported — interpret with care:")
        display(weak)
else:
    print("No X-prefixed imputation flags accompany this file.")

No X-prefixed imputation flags accompany this file.


## 8. Decode categoricals

Labels from the published value sets, not hand-typed mappings.

In [8]:
CATEGORICALS = []

unresolved = {}
for col in CATEGORICALS:
    if col in frame.columns:
        frame = iu.decode(frame, valuesets, col)
        unmatched = frame.loc[frame[col].notna() & frame[f"{col}_LABEL"].isna(), col].unique()
        if len(unmatched):
            unresolved[col] = sorted(unmatched.tolist())[:10]

# An unmatched code means a taxonomy change or a parsing fault. Either way the
# labels are wrong, so this stops the notebook rather than printing a warning.
assert not unresolved, f"codes absent from the published value set: {unresolved}"

label_cols = [c for c in frame.columns if c.endswith("_LABEL")]
frame[CATEGORICALS + label_cols].drop_duplicates().head(20) if label_cols else frame.head()

,UNITID
0,100654
1,100663
2,100706
3,100724
4,100733


## 9. Reshape to the declared grain

Target grain: `UNITID`. The grain is asserted, not assumed, because a duplicated key silently inflates every aggregate computed downstream.

In [9]:
curated = frame.copy()

# Finance is the one component that cannot be a pass-through. Three mutually
# incompatible forms cover the sector and must be harmonised onto shared concepts
# before anything can be compared across it:
#
#   F1A -> GASB public institutions
#   F2  -> FASB private not-for-profit (plus a few FASB-reporting publics)
#   F3  -> FASB private for-profit
#
# Every source column below was confirmed against the three FY2023 dictionaries.
#
# Totals come from the revenue and expense detail parts, not the summary parts. For
# FASB filers the summary totals (F2B01/F3B01, "Total revenues and investment return")
# sit in Parts A-B, which a parent institution reports for itself plus ALL its child
# campuses, while children leave them blank. In FY2023 that blanks F3B01 for 1,592 of
# 2,090 for-profit filers, and for parents it puts a system-wide denominator under a
# campus-level tuition numerator. F1D01/F1D02 (GASB Part D) show the same gap: 429 of
# 1,916 blank. The detail totals below are complete for every filer.
# Re-confirm them against cell 4 before reusing this map for another cycle: the forms
# are revised independently of one another, so a column that moved on one form will
# silently mismatch the others.

CONCEPTS = {
    "F1A": {
        "TOTAL_REVENUES": "F1B27",       # Total operating and nonoperating revenues (Part B)
        "TOTAL_EXPENSES": "F1C191",      # Total expenses and deductions, current year (Part C)
        "TUITION_REVENUE": "F1B01",      # Tuition and fees, net of discounts
        "INSTRUCTION_EXPENSE": "F1C011", # Instruction, current year total
        "ENDOWMENT_EOY": "F1H02",        # Endowment assets, end of fiscal year
    },
    "F2": {
        "TOTAL_REVENUES": "F2D16",       # Total revenues and investment return (Part D)
        "TOTAL_EXPENSES": "F2E131",      # Total expenses, total amount (Part E)
        "TUITION_REVENUE": "F2D01",
        "INSTRUCTION_EXPENSE": "F2E011",
        "ENDOWMENT_EOY": "F2H02",
    },
    "F3": {
        "TOTAL_REVENUES": "F3D09",       # Total revenues and investment return (Part D)
        "TOTAL_EXPENSES": "F3E071",      # Total expenses, total amount (Part E)
        "TUITION_REVENUE": "F3D01",
        "INSTRUCTION_EXPENSE": "F3E011",
        "ENDOWMENT_EOY": None,  # For-profit institutions report no endowment.
    },
}

harmonised = []
for form, table in (("F1A", "F2223_F1A"), ("F2", "F2223_F2"), ("F3", "F2223_F3")):
    record = next(r for r in provenance if r["table"] == table)
    part = iu.read_csv(record["data_path"])

    out = pd.DataFrame({"UNITID": part["UNITID"]})
    out["REPORTING_STANDARD"] = form
    for concept, column in CONCEPTS[form].items():
        if column is None:
            out[concept] = np.nan          # structurally absent, not missing data
        elif column in part.columns:
            values = pd.to_numeric(part[column], errors="coerce")
            out[concept] = values.mask(values.isin(RESERVED))
        else:
            out[concept] = np.nan
            print(f"  {form}: source column {column} for {concept} absent this cycle")
    harmonised.append(out)
    print(f"{form:4s} {table:12s} {len(part):5,d} institutions")

curated = pd.concat(harmonised, ignore_index=True)

# An institution must appear on exactly one form. Overlap is a routing error, and it
# would double-count that institution in every sector aggregate downstream.
overlap = curated.UNITID[curated.UNITID.duplicated()].unique()
print()
print(f"{len(curated):,} rows, {len(overlap)} institutions on more than one form")
if len(overlap):
    display(curated[curated.UNITID.isin(overlap)].sort_values("UNITID").head(10))
    curated = curated.drop_duplicates(subset=["UNITID"], keep="first")

print()
print("Share non-null by reporting standard:")
display(
    curated.groupby("REPORTING_STANDARD")[list(CONCEPTS["F1A"].keys())]
    .apply(lambda g: g.notna().mean().round(3))
)
print(
    "ENDOWMENT_EOY is empty for F3 by construction: for-profit institutions do not "
    "report endowment assets. That is a structural absence, not missing data, and it "
    "must not be imputed."
)

present_grain = [g for g in GRAIN if g in curated.columns]
duplicated = curated.duplicated(subset=present_grain, keep=False).sum()
print(f"grain {present_grain} -> {len(curated):,} rows, {duplicated} duplicated")
assert duplicated == 0, "Declared grain is not unique; resolve before continuing."

curated.head()

F1A  F2223_F1A    1,916 institutions


F2   F2223_F2     1,766 institutions
F3   F2223_F3     2,090 institutions

5,772 rows, 0 institutions on more than one form

Share non-null by reporting standard:


,TOTAL_REVENUES,TOTAL_EXPENSES,TUITION_REVENUE,INSTRUCTION_EXPENSE,ENDOWMENT_EOY
REPORTING_STANDARD,,,,,
F1A,1.0,1.0,1.0,1.0,0.693
F2,1.0,1.0,1.0,1.0,0.746
F3,1.0,1.0,1.0,1.0,0.000


ENDOWMENT_EOY is empty for F3 by construction: for-profit institutions do not report endowment assets. That is a structural absence, not missing data, and it must not be imputed.
grain ['UNITID'] -> 5,772 rows, 0 duplicated


,UNITID,REPORTING_STANDARD,TOTAL_REVENUES,TOTAL_EXPENSES,TUITION_REVENUE,INSTRUCTION_EXPENSE,ENDOWMENT_EOY
0,100654,F1A,232809852,227586697,51735563,32549205,NaN
1,100663,F1A,5042672296,4677414873,245459954,349881220,739372914.0
2,100706,F1A,330325927,306323891,75728883,72583420,99627019.0
3,100724,F1A,181093907,161817460,38285863,43009909,118616285.0
4,100733,F1A,25837619,44028824,0,0,NaN


## 10. Validate

Rules are declarative so the output is a persistable report: which checks ran, which failed, on how many rows, and which institutions were implicated. That report is the artefact you cite when claiming this table is fit for analysis.

In [10]:
RULES = [
    iu.unique_key('UNITID'),
    iu.not_null('REPORTING_STANDARD'),
    iu.in_range('TOTAL_REVENUES', 0, None, severity='warn'),
    iu.in_range('TOTAL_EXPENSES', 0, None, severity='warn'),
]

report = iu.validate(curated, RULES, SLUG)
report.save(f"../reports/validation/{SLUG}.json")
display(report.to_frame()[["name", "status", "n_offending", "share", "note"]])

print("PASSED" if report.ok else "FAILED")
report.raise_if_failed()

,name,status,n_offending,share,note
0,unique_key(UNITID),pass,0,0.00000,Declared grain must be unique
1,not_null(REPORTING_STANDARD),pass,0,0.00000,Key columns must be populated
2,"in_range(TOTAL_REVENUES,0,None)",fail,1,0.00017,Value plausibility bound
3,"in_range(TOTAL_EXPENSES,0,None)",pass,0,0.00000,Value plausibility bound


PASSED


Report(table='c10_f', rows=5772, results=[{'name': 'unique_key(UNITID)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Declared grain must be unique', 'status': 'pass'}, {'name': 'not_null(REPORTING_STANDARD)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Key columns must be populated', 'status': 'pass'}, {'name': 'in_range(TOTAL_REVENUES,0,None)', 'severity': 'warn', 'n_offending': 1, 'share': 0.00017, 'sample_unitids': ['120166'], 'note': 'Value plausibility bound', 'status': 'fail'}, {'name': 'in_range(TOTAL_EXPENSES,0,None)', 'severity': 'warn', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Value plausibility bound', 'status': 'pass'}], generated_utc='2026-09-24T17:19:33+00:00')

## 11. Write the curated table

The sidecar carries the reference period and grain with the data. This is the defence against assembling a panel by filename year when the underlying periods are offset differently per component.

In [11]:
path = iu.write_curated(
    curated,
    SLUG,
    root="../data/curated",
    reference_period=REFERENCE_PERIOD,
    grain=GRAIN,
    provenance=provenance,
    notes='The same economic concept has different variable names and sometimes different definitions on each form. Comparing a GASB expense total directly against a FASB one is the single most common serious error in IPEDS finance work. Prefer the derived file DRVF2023 for cross-sector comparison, because NCES has already done the harmonisation there, and use the raw forms when a specific line item is needed that the derived file does not carry.',
)

iu.write_provenance(provenance, f"../docs/provenance/{SLUG}.json")
print("wrote", path, f"({len(curated):,} rows x {curated.shape[1]} columns)")

wrote ../data/curated/c10_f.parquet (5,772 rows x 7 columns)


## 12. Exercises

1. Re-run this notebook against the prior collection cycle by changing `TABLES`. The schema lock and the period assertion will both object; resolve each objection and record what changed between cycles.
2. Identify the three columns with the lowest reported-flag share, and argue whether each belongs in a predictive model at all.
3. Construct one derived cross-tabulation from this table, then apply `iu.suppress` and `iu.k_anonymity` to it. Report the smallest equivalence class before and after coarsening, and state the k you would require before publishing.
4. The same economic concept has different variable names and sometimes different definitions on each form. Write a validation rule that would catch this error if a colleague made it, and add it to `RULES` above.